# SGCA Cross-Attention Training Notebook

This notebook documents the compact deterministic SGCA Cross-Attention variant used as the second final model in the manuscript comparison. The model shares the species-gated cross-attention concept but uses a lighter disease head than Unified SGCA.


## Publication Training Discipline

The Cross-Attention checkpoint is trained under the same 384 px internet-robust evaluation protocol used for the final comparison. Checkpoint selection is based on validation macro-F1. Development-external evaluation is reported after model selection and should be interpreted as robustness testing under domain shift.


In [ ]:
from pathlib import Path
import os
import json

# In WSL, launch Jupyter with this environment variable before starting the kernel:
# LD_LIBRARY_PATH=/usr/lib/wsl/lib ./.venv-wsl/bin/jupyter lab
os.environ.setdefault('LD_LIBRARY_PATH', '/usr/lib/wsl/lib')

import torch
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

from sgca.models import (
    MODEL_REGISTRY,
    MODEL_DISPLAY_NAMES,
    build_dataloaders,
    build_dataloaders_from_split_dirs,
    build_evaluation_loader,
    index_species_disease_dataset,
    seed_everything,
    train_model,
    evaluate,
    mc_dropout_predict,
    get_probabilities,
    conformal_prediction_sets,
    load_checkpoint,
    run_single_inference,
    save_reports,
)

print('Available models:', list(MODEL_REGISTRY.keys()))
print('Torch:', torch.__version__)
print('CUDA visible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU not visible in this kernel. Restart Jupyter from WSL with:')
    print('LD_LIBRARY_PATH=/usr/lib/wsl/lib ./.venv-wsl/bin/jupyter lab')


## Training Discipline

Run training with `EVALUATE_DEVELOPMENT_EXTERNAL = False`. Select the best epoch by validation disease macro-F1. Only after that, evaluate the development external cohort once, preferably with `evaluate_hierarchical_decoding.py` so raw, predicted-species constrained, and known-species constrained decoding are reported in the same format as the Unified model.

This is the original CrossAttention architecture, not the newer `SGCA_CrossAttention_Uncertainty_GradCAM` hybrid. It is useful for testing whether the architecture-novelty model benefits from the same 384px internet-robust retraining recipe.


In [ ]:
# Original CrossAttention internet-robust retraining configuration
SPLIT_ROOT = Path('data/training_data_deduped_splits/seed42')
DATA_DIR = Path('training_data')  # fallback only; the clean workflow uses SPLIT_ROOT
EXTERNAL_ROOT = Path('data/development_external_2026_05_02')
RESULTS_DIR = Path('results/models/sgca_cross_attention')
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

# Higher resolution keeps more lesion detail. EfficientNetV2-S native resolution is 384.
# If GPU memory is not enough, drop BATCH_SIZE to 6 or 4 first; keep IMG_SIZE=384 if possible.
IMG_SIZE = 384
BATCH_SIZE = 8
EPOCHS = 50
LR = 2e-5
WEIGHT_DECAY = 1e-4
SEED = 42
NUM_WORKERS = 4
PRETRAINED = True
MC_PASSES = 20

# Original CrossAttention comparison setting: no class-balanced loss, no MixStyle, no AugMix stack.
# The recipe matches the recent Unified retrain so the comparison is controlled.
MODEL_KEY = 'sgca_cross_attention'
AUGMENTATION_POLICY = 'internet_robust'  # options: standard, robust, internet_robust, augmix
CLASS_BALANCED_LOSS = False
CB_BETA = 0.9999
LABEL_SMOOTHING = 0.0
MIXSTYLE_P = 0.0
MIXSTYLE_ALPHA = 0.3
USE_WEIGHTED_SAMPLER = False
EARLY_STOPPING_PATIENCE = 10
EVALUATE_DEVELOPMENT_EXTERNAL = False  # exploratory Plan A; flip to True ONCE after selecting the best checkpoint

seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
print('Device:', DEVICE)
print('AMP enabled:', USE_AMP)
print('Results:', RESULTS_DIR)

In [ ]:
# Build clean deduplicated dataset and loaders
data = build_dataloaders_from_split_dirs(
    train_dir=SPLIT_ROOT / 'train',
    val_dir=SPLIT_ROOT / 'val',
    test_dir=SPLIT_ROOT / 'test',
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    augmentation_policy=AUGMENTATION_POLICY,
    use_weighted_sampler=USE_WEIGHTED_SAMPLER,
)

config = {
    'img_size': IMG_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'seed': SEED,
    'split_root': str(SPLIT_ROOT),
    'external_root': str(EXTERNAL_ROOT),
    'model_key': MODEL_KEY,
    'augmentation_policy': AUGMENTATION_POLICY,
    'class_balanced_loss': CLASS_BALANCED_LOSS,
    'cb_beta': CB_BETA,
    'label_smoothing': LABEL_SMOOTHING,
    'mixstyle_p': MIXSTYLE_P,
    'mixstyle_alpha': MIXSTYLE_ALPHA,
    'species_names': data.species_names,
    'disease_names': data.disease_names,
    'train_size': len(data.train_df),
    'val_size': len(data.val_df),
    'test_size': len(data.test_df),
}

with open(RESULTS_DIR / 'dataset_config.json', 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2)

print('Species:', data.species_names)
print('Number of diseases:', len(data.disease_names))
print('Clean split sizes:', len(data.train_df), len(data.val_df), len(data.test_df))


In [ ]:
# Quick CrossAttention sanity check
model_cls = MODEL_REGISTRY[MODEL_KEY]
model = model_cls(
    len(data.species_names),
    len(data.disease_names),
    pretrained=False,
    mixstyle_p=MIXSTYLE_P,
    mixstyle_alpha=MIXSTYLE_ALPHA,
).to(DEVICE)
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    outputs = model(dummy)
species_out, disease_out = outputs[:2] if isinstance(outputs, tuple) else outputs
params = sum(p.numel() for p in model.parameters())
print(f'{MODEL_KEY:32s} species={tuple(species_out.shape)} disease={tuple(disease_out.shape)} params={params:,}')
del model, dummy
if DEVICE.type == 'cuda':
    torch.cuda.empty_cache()


In [ ]:
# Train the original SGCA CrossAttention variant
model, history_df, best_path = train_model(
    model_key=MODEL_KEY,
    data=data,
    device=DEVICE,
    results_dir=RESULTS_DIR,
    epochs=EPOCHS,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    pretrained=PRETRAINED,
    use_amp=USE_AMP,
    class_balanced_loss=CLASS_BALANCED_LOSS,
    cb_beta=CB_BETA,
    label_smoothing=LABEL_SMOOTHING,
    mixstyle_p=MIXSTYLE_P,
    mixstyle_alpha=MIXSTYLE_ALPHA,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
)

print('Best checkpoint:', best_path)
history_df.tail()


In [ ]:
# Evaluate validation/internal clean test, and only optionally the development external set
val_metrics = evaluate(model, data.val_dl, DEVICE)
test_metrics = evaluate(model, data.test_dl, DEVICE)

print('Validation disease acc:', round(val_metrics['disease_acc'], 4))
print('Validation disease macro-F1:', round(val_metrics['disease_f1'], 4))
print('Internal clean test disease acc:', round(test_metrics['disease_acc'], 4))
print('Internal clean test disease macro-F1:', round(test_metrics['disease_f1'], 4))

save_reports(val_metrics, data.species_names, data.disease_names, RESULTS_DIR / 'val')
save_reports(test_metrics, data.species_names, data.disease_names, RESULTS_DIR / 'internal_test')

external_metrics = None
external_present_f1 = None
external_df = None
if EVALUATE_DEVELOPMENT_EXTERNAL:
    external_frame = index_species_disease_dataset(EXTERNAL_ROOT)
    external_loader, external_df = build_evaluation_loader(
        external_frame,
        img_size=IMG_SIZE,
        species_names=data.species_names,
        disease_names=data.disease_names,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
    )
    external_metrics = evaluate(model, external_loader, DEVICE)
    external_present_labels = sorted(set(external_metrics['y_dis'].tolist()))
    external_present_f1 = f1_score(
        external_metrics['y_dis'],
        external_metrics['p_dis'],
        labels=external_present_labels,
        average='macro',
        zero_division=0,
    )
    print('Development external images:', len(external_df))
    print('Development external disease acc:', round(external_metrics['disease_acc'], 4))
    print('Development external present-class macro-F1:', round(float(external_present_f1), 4))
    print('Development external sklearn-default macro-F1:', round(external_metrics['disease_f1'], 4))
    save_reports(external_metrics, data.species_names, data.disease_names, RESULTS_DIR / 'development_external')
else:
    print('Development external evaluation skipped. Set EVALUATE_DEVELOPMENT_EXTERNAL=True only after selecting the final checkpoint.')

In [ ]:
# MC-Dropout uncertainty for models with dropout-enabled heads
if MODEL_KEY in {'unified_sgca_uncertainty', 'vetderm_hat', 'sgca_cross_attention'}:
    mc = mc_dropout_predict(model, data.test_dl, device=DEVICE, passes=MC_PASSES)
    mc_acc = accuracy_score(mc['labels'], mc['preds'])
    mc_f1 = f1_score(mc['labels'], mc['preds'], average='macro', zero_division=0)
    print('MC-Dropout accuracy:', round(float(mc_acc), 4))
    print('MC-Dropout macro-F1:', round(float(mc_f1), 4))
    print('Mean entropy:', round(float(mc['pred_entropy'].mean()), 4))
    print('Mean mutual information:', round(float(mc['mutual_info'].mean()), 4))
else:
    print('MC-Dropout analysis is intended for dropout-enabled models.')

In [ ]:
# Conformal prediction on validation/test probabilities
probs_val, labels_val, _ = get_probabilities(model, data.val_dl, DEVICE)
probs_test, labels_test, _ = get_probabilities(model, data.test_dl, DEVICE)

cp = conformal_prediction_sets(
    probs_cal=probs_val,
    labels_cal=labels_val,
    probs_test=probs_test,
    labels_test=labels_test,
    alpha=0.1,
)

print('Conformal coverage:', round(cp.get('coverage', 0.0), 4))
print('Mean set size:', round(cp['mean_set_size'], 4))
print('Singleton rate:', round(cp['singleton_rate'], 4))

In [ ]:
# Optional: print the equivalent command-line runner call for this single CrossAttention internet-robust variant.
# Keep --evaluate-external off during training; use it only after the checkpoint is selected by validation metrics.
RUN_VARIANT_COMMAND = False

if RUN_VARIANT_COMMAND:
    cmd = (
        'LD_LIBRARY_PATH=/usr/lib/wsl/lib ./.venv-wsl/bin/python run_external_generalization_experiment.py '
        '--variants crossattention_internet_robust '
        f'--epochs {EPOCHS} --batch-size {BATCH_SIZE} --img-size {IMG_SIZE} '
        f'--lr {LR} --weight-decay {WEIGHT_DECAY} --early-stopping-patience {EARLY_STOPPING_PATIENCE} '
        f'--results-dir {RESULTS_DIR.parent / "runner_sgca_crossattention_internet_robust_384_seed42"}'
    )
    print(cmd)
    print('After the best epoch is selected by validation F1, evaluate the chosen checkpoint once.')
else:
    print('Set RUN_VARIANT_COMMAND=True to print the equivalent single-variant retraining command.')

In [ ]:
# Save notebook summary row
summary_csv = RESULTS_DIR / 'notebook_external_generalization_summary.csv'

row = {
    'model_key': MODEL_KEY,
    'model_name': MODEL_DISPLAY_NAMES[MODEL_KEY],
    'best_checkpoint': str(best_path),
    'augmentation_policy': AUGMENTATION_POLICY,
    'class_balanced_loss': CLASS_BALANCED_LOSS,
    'label_smoothing': LABEL_SMOOTHING,
    'mixstyle_p': MIXSTYLE_P,
    'mixstyle_alpha': MIXSTYLE_ALPHA,
    'val_species_acc': round(val_metrics['species_acc'], 4),
    'val_species_f1': round(val_metrics['species_f1'], 4),
    'val_disease_acc': round(val_metrics['disease_acc'], 4),
    'val_disease_f1': round(val_metrics['disease_f1'], 4),
    'test_species_acc': round(test_metrics['species_acc'], 4),
    'test_species_f1': round(test_metrics['species_f1'], 4),
    'test_disease_acc': round(test_metrics['disease_acc'], 4),
    'test_disease_f1': round(test_metrics['disease_f1'], 4),
}

if external_metrics is not None:
    row.update({
        'external_species_acc': round(external_metrics['species_acc'], 4),
        'external_species_f1': round(external_metrics['species_f1'], 4),
        'external_disease_acc': round(external_metrics['disease_acc'], 4),
        'external_disease_f1_present': round(float(external_present_f1), 4),
        'external_disease_f1_sklearn_default': round(external_metrics['disease_f1'], 4),
    })

if 'mc' in globals():
    row['mc_dropout_acc'] = round(float(mc_acc), 4)
    row['mc_dropout_f1'] = round(float(mc_f1), 4)
    row['mean_entropy'] = round(float(mc['pred_entropy'].mean()), 4)
    row['mean_mutual_info'] = round(float(mc['mutual_info'].mean()), 4)

if 'cp' in globals():
    row['conformal_coverage'] = round(float(cp.get('coverage', 0.0)), 4)
    row['conformal_mean_set_size'] = round(float(cp['mean_set_size']), 4)
    row['conformal_singleton_rate'] = round(float(cp['singleton_rate']), 4)

if summary_csv.exists():
    df_summary = pd.read_csv(summary_csv)
    df_summary = pd.concat([df_summary, pd.DataFrame([row])], ignore_index=True)
else:
    df_summary = pd.DataFrame([row])

sort_col = 'external_disease_f1_present' if 'external_disease_f1_present' in df_summary.columns else 'val_disease_f1'
df_summary.to_csv(summary_csv, index=False)
display(df_summary.sort_values(sort_col, ascending=False).reset_index(drop=True))
print('Saved:', summary_csv)


In [ ]:
# Single-image inference. The old CrossAttention helper writes prediction JSON; Grad-CAM artifacts are reserved for the deployable GradCAM model variants.
IMAGE_PATH = ''
CHECKPOINT_PATH = str(best_path) if 'best_path' in globals() else str(RESULTS_DIR / 'sgca_cross_attention_best.pt')

if IMAGE_PATH:
    run_single_inference(
        checkpoint_path=Path(CHECKPOINT_PATH),
        image_path=Path(IMAGE_PATH),
        out_dir=RESULTS_DIR / 'inference_outputs',
        mc_passes=MC_PASSES,
        pretrained=PRETRAINED,
        device=DEVICE,
    )
else:
    print('Set IMAGE_PATH to run inference.')
